# Networking & Edge

## What's covered

- The **path of a request** from a user's browser to your servers and back
- **DNS** — how a name becomes an address, the role of caching and TTLs
- **CDNs** — push vs. pull, when caching at the edge pays for itself
- **Load balancing** — L4 vs L7, the common algorithms, consistent hashing
- **API design** — REST vs gRPC vs GraphQL, and when each one fits
- **Rate limiting** — token bucket, leaky bucket, fixed and sliding windows


## The path of a request

Before getting to specific components, fix the path a single HTTP request takes. Every box in the diagram is a system you can design or break:

```
   browser
      |
      | (1) DNS lookup — "api.example.com" -> 203.0.113.5
      v
   CDN edge POP (often) — cached static assets served here
      |
      | (2) if not cached, forward toward origin
      v
   load balancer at the edge
      |
      | (3) pick a healthy server, route there
      v
   application server (one of N behind the LB)
      |
      | (4) reads, writes, fans out to downstream services
      v
   database, cache, message queue, ...
```

Each hop adds latency. The latency table from notebook one is the budget: same-datacenter round-trip is half a millisecond, cross-continent is forty milliseconds, cross-world is a hundred fifty. A request that crosses three of those hops sequentially can spend more time on the wire than in code. The job of this layer is to **shorten the wire path** (CDN, edge POPs), **avoid wasted hops** (smart routing, connection reuse), and **fail gracefully** when a downstream is slow.


## DNS — turning names into addresses

DNS is the distributed phone book of the internet. The browser needs an IP address to open a TCP connection; the user typed a name. DNS bridges the gap.

**The resolution chain.** A name like `api.example.com` resolves through up to four levels:

1. **Recursive resolver** — usually run by your ISP or a public service (8.8.8.8, 1.1.1.1). It does the heavy lifting and caches results.
2. **Root nameservers** — return the address of the right TLD server (".com" lives over there).
3. **TLD nameservers** — return the address of the authoritative server for `example.com`.
4. **Authoritative nameserver** — for `example.com`. Returns the actual address for `api.example.com`.

A fully cold lookup touches all four. With caching at every level, the typical lookup is a single round-trip to the resolver, which has the answer cached.

**TTL — time to live.** Every DNS record has a TTL telling caches how long to keep it. Short TTLs (30-300 seconds) let you change routing fast — useful for failover. Long TTLs (hours to days) reduce DNS query load and protect against resolver outages. The trade-off is real: a one-hour TTL means a one-hour delay before a DNS-based failover takes effect everywhere. Production services tend to run 30-60 second TTLs for records that might need to change quickly, and several hours for static records.

**Anycast.** A single IP address advertised from many physical locations. BGP routes each user's packets to the *nearest* location. The recursive resolvers `8.8.8.8` and `1.1.1.1` are both anycast — there is no single server at those addresses; there are dozens, one per region. CDN edge POPs use the same trick. Anycast turns "send the request to the nearest server" into a routing problem solved by the network, not by application code.

**DNS load balancing.** Returning multiple A records for one name (round-robin DNS) is the cheapest form of load balancing, and the worst — clients cache the answer, resolvers cache the answer, you cannot weight by health or capacity, and removing a sick server takes one full TTL to propagate. Useful as a coarse layer; never as your only layer.


## CDN — caching at the edge

A **Content Delivery Network** is a distributed network of cache servers placed close to users — physically close, in major cities and ISPs. When a user requests an asset, the request hits the nearest CDN edge POP (Point of Presence). If the asset is cached there, the user gets it in 5-20ms instead of 100-200ms.

**What goes on a CDN:** static assets (images, JavaScript, CSS, videos, downloads), but also increasingly dynamic content with short TTLs ("edge caching of JSON responses for 60 seconds"). The decision criterion is **how much staleness can the content tolerate** — anything that's the same for everyone and changes infrequently is a CDN win.

**Push vs pull.**

- **Pull CDN** (the default, the common case) — the CDN fetches content from the origin on the first request, caches it, and serves it from cache for the rest of the TTL. Simple to operate: you change your origin, the CDN catches up eventually.
- **Push CDN** — you upload content to the CDN explicitly. Used for large files where you don't want the first request to bear the origin-fetch latency, or for content that isn't safely reachable from the public internet.

Most modern CDNs (Cloudflare, Fastly, AWS CloudFront) are primarily pull with optional pre-warming.

**Cache invalidation — the hard problem.** When the underlying content changes, the CDN's cached copy is now stale. Three common approaches:

- **TTL expiry** — wait. Cheapest, but staleness can last as long as the TTL.
- **Purge** — explicitly tell the CDN "invalidate this URL". Reliable but rate-limited and slow (seconds to minutes globally).
- **Versioned URLs** — change the URL when the content changes. `app.v23.css` instead of `app.css`. The CDN sees a new URL, fetches the new content, the old URL keeps serving the old version (which nobody requests anymore). This is the gold standard for static assets and what every modern build tool does by default.

**Why CDN edge compute matters.** Newer CDNs run code at the edge (Cloudflare Workers, Lambda@Edge, Fastly Compute). Authentication checks, A/B test routing, simple personalization can run within 5-10ms of the user, never hitting the origin. This is where the *boundary between CDN and application server* is currently shifting.


## Load balancing — spreading the work

A load balancer sits between clients and your fleet of servers. It picks a healthy server, forwards the request, returns the response. From the client's view, the LB is the service.

**Two questions to ask about any load balancer:**

- **What layer does it work at?** Layer 4 (transport) or Layer 7 (application).
- **How does it pick a server?** The algorithm — round-robin, least connections, consistent hashing, something else.

### L4 vs L7

**L4 load balancers** route at the TCP/UDP level. They see the source/destination IP and port; they don't see HTTP, headers, paths, or cookies. Fast (no parsing, no termination), but limited to coarse routing. Examples: AWS NLB, HAProxy in TCP mode, IPVS.

**L7 load balancers** terminate the TCP connection, parse the HTTP request, and route based on the request itself — path, headers, host, cookie. Slower (parses every request), but lets you do path-based routing, header-based A/B routing, sticky sessions by cookie. Examples: AWS ALB, NGINX, Envoy, HAProxy in HTTP mode.

**Which to pick:**

- L4 for raw throughput, TCP-level services (databases, custom protocols, very high QPS HTTP where every microsecond matters).
- L7 for HTTP services that need any of: path routing, host routing, cookies, header inspection, TLS termination at the LB, request mirroring, response rewriting.

Most modern web services run L7. Most internal infra (database proxies, RPC) runs L4.


### Load balancing algorithms

How does the LB pick which server to forward the request to?

- **Round-robin** — server 1, then 2, then 3, then back to 1. Simple, ignores load. Good when all servers are identical and all requests are equal-cost. Bad when either is false.
- **Least connections** — pick the server with the fewest active connections. Adapts to slow requests automatically (slow requests hold connections; that server gets fewer new ones). Good default for variable-cost requests.
- **Weighted round-robin / weighted least-connections** — different weights per server. Useful during a deploy (give the new servers less traffic until they warm up) or for heterogeneous fleets.
- **IP hash / sticky sessions** — route the same client to the same server every time. Used when the server caches per-session state. Fragile — losing the server loses the session — and discouraged in modern designs in favour of stateless servers with external session storage.
- **Consistent hashing** — hash the request key (a user ID, a cache key) to pick the server. Adding or removing a server only reshuffles a small fraction of keys, not all of them. Essential for cache shards, message brokers, distributed databases.

We'll work through consistent hashing in code, because it shows up in every distributed system in this series.


### Consistent hashing — the algorithm


In [ ]:
# Consistent hashing — the classic distributed-systems algorithm.
# Each server (and each replica of it) lives on a circular hash space.
# A key is placed by hashing it, then walking clockwise to the next server.
# Adding/removing a server only moves the keys between it and its neighbour.

import hashlib
from bisect import bisect_right, insort


class ConsistentHashRing:
    def __init__(self, virtual_nodes_per_server: int = 100):
        # virtual nodes give better distribution — each server occupies many
        # points on the ring, so removing one server smears its keys across
        # the rest rather than handing them all to one neighbour.
        self.vnodes = virtual_nodes_per_server
        self._ring: list[int] = []           # sorted hash positions
        self._by_pos: dict[int, str] = {}    # position -> server name

    def _hash(self, key: str) -> int:
        return int(hashlib.md5(key.encode()).hexdigest(), 16) % (2 ** 32)

    def add_server(self, name: str) -> None:
        for i in range(self.vnodes):
            pos = self._hash(f"{name}#{i}")
            insort(self._ring, pos)
            self._by_pos[pos] = name

    def remove_server(self, name: str) -> None:
        positions_to_remove = [p for p, n in self._by_pos.items() if n == name]
        for p in positions_to_remove:
            self._ring.remove(p)
            del self._by_pos[p]

    def server_for(self, key: str) -> str:
        if not self._ring:
            raise RuntimeError("empty ring")
        pos = self._hash(key)
        i = bisect_right(self._ring, pos) % len(self._ring)  # walk clockwise
        return self._by_pos[self._ring[i]]


# Demonstrate the property: adding a server moves only ~1/N of keys.
KEYS = [f"user-{i}" for i in range(10_000)]

ring = ConsistentHashRing(virtual_nodes_per_server=100)
for s in ["s1", "s2", "s3", "s4"]:
    ring.add_server(s)

before = {k: ring.server_for(k) for k in KEYS}

ring.add_server("s5")
after = {k: ring.server_for(k) for k in KEYS}

moved = sum(1 for k in KEYS if before[k] != after[k])
print(f"keys total: {len(KEYS)}")
print(f"keys moved when adding s5: {moved} ({moved/len(KEYS):.1%})")
print(f"expected for 5 servers:    ~{len(KEYS)//5} ({1/5:.0%})")

# Distribution check.
from collections import Counter
counts = Counter(after.values())
for s, n in sorted(counts.items()):
    print(f"  {s}: {n:5d} keys ({n/len(KEYS):.1%})")


The key property: adding a fifth server reshuffles about a fifth of the keys (1/N), *not* all of them. Naïve modulo hashing (`hash(key) % N`) reshuffles almost everything when N changes — catastrophic if those keys are cache entries (mass cache miss) or shard locations (mass data migration). Consistent hashing is what makes cache fleets, sharded databases, and distributed-hash-table systems like DynamoDB and Cassandra tolerable to resize.


## API design — REST vs gRPC vs GraphQL

Three dominant styles for designing the contract between a client and a server. Each makes different trade-offs.

### REST

The default for public APIs. HTTP verbs (GET, POST, PUT, DELETE) act on **resources** identified by URLs. Bodies are usually JSON.

- **Pros:** universally understood, debuggable with a browser or curl, cacheable at the HTTP layer (CDNs love GETs), works through any firewall, easy to evolve (add new fields, ignore unknown ones).
- **Cons:** verbose on the wire (JSON + HTTP headers), no schema enforced by default (OpenAPI is opt-in), over- and under-fetching (the response is whatever the endpoint returns, not what the client wants), every endpoint is its own contract.

**Use REST for:** public APIs, browser-served APIs, low-to-moderate QPS internal services where developer ergonomics matter more than wire efficiency.

### gRPC

Google's RPC framework, built on HTTP/2 and Protocol Buffers (protobuf). The contract is a `.proto` file; client and server code are generated from it.

- **Pros:** strict schema, compact binary wire format (typically 5-10x smaller than JSON), bidirectional streaming, generated clients in every major language, HTTP/2 multiplexing gives much better connection reuse than REST.
- **Cons:** opaque on the wire (binary, you need the proto to decode), needs HTTP/2 (some proxies and browsers struggle), not natively cacheable by HTTP caches, harder to debug, browser support requires gRPC-Web or a proxy.

**Use gRPC for:** internal service-to-service communication, high-QPS or low-latency-budget paths, polyglot fleets where one schema generates clients for Go, Java, Python, and Rust.

### GraphQL

A query language where the *client* declares what fields it needs and the server returns exactly that. One endpoint, one POST, one flexible query.

- **Pros:** clients fetch exactly what they need (no over-fetch, no under-fetch), one round-trip for what would be many in REST, strong schema, great for UIs with deeply nested data needs.
- **Cons:** harder to cache (every query is different), harder to rate-limit (one query can be cheap or arbitrarily expensive), N+1 query patterns are easy to introduce, server-side complexity is significant, can be hard to debug at scale.

**Use GraphQL for:** UI-facing APIs where one screen pulls from many resources, mobile clients on slow networks (round-trip savings matter), apps with rapidly evolving frontends and a stable backend.

### Picking between them

A reasonable default:

- **Public-facing API** → REST. Universal understanding wins.
- **Internal service-to-service** → gRPC. Schema and wire efficiency win.
- **Client-driven aggregation** (mobile app, complex UI) → GraphQL, often as a Backend-for-Frontend layer in front of REST or gRPC.

Many real systems use all three: REST for the public boundary, GraphQL for the BFF layer, gRPC between services.


## Rate limiting — protecting the system from itself

Rate limiting caps the rate at which a client can make requests. It exists for three reasons:

1. **Fair use.** One abusive client cannot starve others of capacity.
2. **Cost control.** Each request costs CPU, memory, downstream calls. Unbounded is unaffordable.
3. **Protection.** A bug, a runaway script, or a deliberate attack cannot bring the system down.

Rate limits are usually expressed as **N requests per time window per key**. The key might be a user ID, an API key, a source IP, or a (user, endpoint) pair. The hard part is *how to enforce the limit accurately* with low overhead, especially across a fleet of servers.

**Four common algorithms:**

- **Fixed window** — count requests in a clock-aligned bucket (e.g. requests in the current minute). Simple. Suffers from boundary spikes — a client can hit the limit at the end of one window and again at the start of the next, doubling the effective rate.
- **Sliding window log** — keep a timestamp for each request, count how many are within the last N seconds. Accurate. Memory cost is O(requests-per-window) per key — expensive at high limits.
- **Sliding window counter** — approximate the sliding window using two adjacent fixed windows weighted by overlap. Cheap memory, almost as accurate as a real sliding window. The right choice when you can't afford the log.
- **Token bucket** — a bucket holds N tokens, refills at R tokens per second. Each request consumes one token. Allows bursts up to N, then settles to the steady rate R. The canonical algorithm — what most cloud rate limiters use internally.

The trade-off pattern: simpler algorithms are cheaper but allow more burstiness or boundary cheating. Token bucket is the sweet spot for most applications.


### Token bucket — the algorithm


In [ ]:
import time
from dataclasses import dataclass


@dataclass
class TokenBucket:
    capacity: float       # max burst size — tokens the bucket can hold
    refill_rate: float    # tokens added per second (the steady-state limit)
    tokens: float = 0.0
    last_refill: float = 0.0

    def __post_init__(self):
        self.tokens = float(self.capacity)
        self.last_refill = time.monotonic()

    def allow(self, cost: float = 1.0) -> bool:
        now = time.monotonic()
        # refill since last check, capped at capacity
        elapsed = now - self.last_refill
        self.tokens = min(self.capacity, self.tokens + elapsed * self.refill_rate)
        self.last_refill = now
        if self.tokens >= cost:
            self.tokens -= cost
            return True
        return False


# Configure: 10 requests/sec steady, allow bursts up to 20.
bucket = TokenBucket(capacity=20, refill_rate=10)

# Slam it with 30 requests instantly.
allowed = sum(1 for _ in range(30) if bucket.allow())
print(f"burst of 30: allowed {allowed} (expected ~20 — the bucket capacity)")

# After 1 second of refill, the bucket has ~10 more tokens.
time.sleep(1.0)
allowed = sum(1 for _ in range(30) if bucket.allow())
print(f"after 1s refill: allowed {allowed} (expected ~10 — the refill rate × 1s)")

# Steady-state demonstration — at 10 tokens/sec, a sustained 100/sec is throttled.
bucket = TokenBucket(capacity=20, refill_rate=10)
allowed_sustained = 0
start = time.monotonic()
while time.monotonic() - start < 0.5:    # half a second
    if bucket.allow():
        allowed_sustained += 1
print(f"sustained over 0.5s: allowed {allowed_sustained}")
print(f"(expected ~25 — the 20 initial tokens, plus ~5 refilled during the 0.5s)")


**Distributed rate limiting** is the next layer of complexity. If you have ten servers, each running an in-process token bucket, your effective limit is ten times whatever you configured. Solutions:

- **Sticky routing** — route a given key consistently to the same server (consistent hashing again). Each server enforces the limit alone. Simple. Breaks under server failures.
- **Centralized counter** — a shared Redis or memcached holding the token bucket state. Every request consults the central store. Accurate but adds a round-trip and concentrates load.
- **Two-tier approximation** — each server has a local approximate counter that periodically syncs with a central authority. Accurate-enough, low-latency. Used by most large-scale services.

The pattern: simple in-process rate limiting is fine for one server. Across a fleet, you need to pick what to give up — strict accuracy, latency, or simplicity. You can't have all three.


Notebook three turns to **Databases & Storage** — the persistence layer underneath every service. Relational versus document versus key-value stores, the indexing structures that make them fast, replication for fault tolerance and read scale, sharding for write scale, and the cache strategies that sit in front of all of it. The latency table from notebook one is, again, the budget every choice has to fit into.
